# Cohere + Mistral + Anthropic tie-break

In [1]:
!pip -q install anthropic pandas tqdm

In [2]:
import os
from getpass import getpass
import json
import re
import time

import pandas as pd
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

from anthropic import Anthropic

import warnings
warnings.simplefilter("ignore", FutureWarning)

os.environ["ANTHROPIC_API_KEY"] = getpass("Anthropic API key: ").strip()
anthropic_client = Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

ANTHROPIC_MODEL = "claude-sonnet-4-5"

In [3]:
cohere_df = pd.read_csv("cohere_fs_annotation.csv", sep=";")
mistral_df = pd.read_csv("mistral_fs_annotation.csv", sep=";")

print(f"Cohere:  {cohere_df.shape}")
print(f"Mistral: {mistral_df.shape}")

mismatch = (cohere_df["text"] != mistral_df["text"]).sum()
print(f"\nСтрок с разным текстом по тому же индексу: {mismatch}")

if mismatch == 0:
    merged = cohere_df.copy()
    merged["mistral_strategy1"]   = mistral_df["mistral_strategy1"]
    merged["mistral_strategy2"]   = mistral_df["mistral_strategy2"]
    merged["mistral_confidence"]  = mistral_df["mistral_confidence"]
    merged["mistral_explanation"] = mistral_df["mistral_explanation"]
    merged["mistral_error"]       = mistral_df["mistral_error"]
else:
    print("Тексты не совпадают построчно")
    mistral_slim = mistral_df[[
        "text", "speaker", "order",
        "mistral_strategy1", "mistral_strategy2",
        "mistral_confidence", "mistral_explanation", "mistral_error",
    ]]
    merged = cohere_df.merge(mistral_slim, on=["text", "speaker", "order"], how="left")

print(f"\nMerged shape: {merged.shape}")

Cohere:  (42284, 15)
Mistral: (42284, 15)

Строк с разным текстом по тому же индексу: 0

Merged shape: (42284, 20)


In [4]:
EMPTY_VALUES = {"", "nan", "none", "null"}

def normalize_strategy_value(x):
    if pd.isna(x):
        return None
    s = str(x).strip().lower()
    if s in EMPTY_VALUES:
        return None
    return s

def pair_equal(s1_a, s2_a, s1_b, s2_b):
    return (normalize_strategy_value(s1_a) == normalize_strategy_value(s1_b) and
            normalize_strategy_value(s2_a) == normalize_strategy_value(s2_b))

In [5]:
agree_s1 = merged.apply(
    lambda r: normalize_strategy_value(r["cohere_strategy1"]) == normalize_strategy_value(r["mistral_strategy1"]),
    axis=1
)
agree_s2 = merged.apply(
    lambda r: normalize_strategy_value(r["cohere_strategy2"]) == normalize_strategy_value(r["mistral_strategy2"]),
    axis=1
)
agree_both = agree_s1 & agree_s2

total = len(merged)
print(f"Всего строк: {total}")
print(f"Согласие по strategy1: {agree_s1.sum()} / {total} = {agree_s1.mean()*100:.2f}%")
print(f"Согласие по strategy2: {agree_s2.sum()} / {total} = {agree_s2.mean()*100:.2f}%")
print(f"Согласие по ОБЕИМ:     {agree_both.sum()} / {total} = {agree_both.mean()*100:.2f}%")

non_mod = merged["cohere_strategy1"].apply(lambda x: str(x).strip() != "-") | \
          merged["mistral_strategy1"].apply(lambda x: str(x).strip() != "-")
print(f"\nНе-модераторских строк: {non_mod.sum()}")
print(f"Согласие по strategy1 (без модер.): {agree_s1[non_mod].mean()*100:.2f}%")
print(f"Согласие по ОБЕИМ (без модер.):     {agree_both[non_mod].mean()*100:.2f}%")

Всего строк: 42284
Согласие по strategy1: 36736 / 42284 = 86.88%
Согласие по strategy2: 39202 / 42284 = 92.71%
Согласие по ОБЕИМ:     34917 / 42284 = 82.58%

Не-модераторских строк: 24025
Согласие по strategy1 (без модер.): 76.91%
Согласие по ОБЕИМ (без модер.):     69.34%


In [6]:
# Confusion matrix по strategy1
def norm_for_display(s):
    n = normalize_strategy_value(s)
    return n if n is not None else "<none>"

c1_disp = merged["cohere_strategy1"].apply(norm_for_display)
m1_disp = merged["mistral_strategy1"].apply(norm_for_display)
pd.crosstab(c1_disp, m1_disp, rownames=["Cohere"], colnames=["Mistral"], margins=True)

Mistral,-,accusation,appeal,no_strategy,presentation,self-justification,All
Cohere,,,,,,,
-,18259,77,25,469,18,17,18865
accusation,1,3795,262,300,1192,140,5690
appeal,0,89,1862,236,663,6,2856
no_strategy,4,88,42,5848,143,25,6150
presentation,0,94,453,143,5767,113,6570
self-justification,0,126,20,292,510,1205,2153
All,18264,4269,2664,7288,8293,1506,42284


In [7]:
disputed = merged[~agree_both].copy().reset_index(drop=True)
print(f"Спорных строк: {len(disputed)} ({len(disputed)/total*100:.1f}% от всех)")

Спорных строк: 7367 (17.4% от всех)


In [8]:
TIE_BREAK_PROMPT = """You are an expert annotator of rhetorical strategies in U.S. presidential and vice-presidential debate transcripts. Two annotators have labeled the following debate fragment differently. Your task is to choose which annotation is more accurate.

## Taxonomy
1. 'presentation' (positive framing, self-presentation) - speaker explicitly represents themselves, their party, their candidate, situation, event, policy outcome, or trend in a positive light. Positive evaluation of a situation counts as presentation only when the speaker associates the positive outcome with their own side.
2. 'accusation' (negative framing of others) - speaker attributes blame or wrongdoing to an opponent, or describes a situation, event, or trend negatively while attributing the cause to a specific actor.
3. 'self-justification' - speaker's main rhetorical move is to deny, explain, or contextualize a specific accusation against themselves or their record.
4. 'appeal' - an audience-oriented rhetorical move: a direct call to the audience to act, vote, support, or adopt a position, or a broader appeal that invokes shared values, identity, or collective responsibility, or emotional storytelling addressed to the audience to mobilize support.

Special labels:
- '-' - moderator or question-asker utterance.
- 'no_strategy' - fragment is too short, interrupted, purely emotional, procedural, or lacks sufficient context to identify a strategy.

## Note on the accusation+presentation pair
When both annotators agree the pair is {accusation, presentation}, the canonical order is strategy1='accusation', strategy2='presentation'.

## Your task
You will be given:
- the debate fragment,
- annotation A (strategy1, strategy2, explanation),
- annotation B (strategy1, strategy2, explanation).

Pick the more accurate annotation as a whole pair. You must choose either "A" or "B" - do not propose a third option.

Return strictly valid JSON, no extra text:
{
  "choice": "A" or "B",
  "explanation": "one sentence explaining your choice"
}
"""

def build_tie_break_user_message(text, a_s1, a_s2, a_expl, b_s1, b_s2, b_expl):
    a_s2 = a_s2 if (a_s2 and str(a_s2).strip().lower() not in EMPTY_VALUES) else "null"
    b_s2 = b_s2 if (b_s2 and str(b_s2).strip().lower() not in EMPTY_VALUES) else "null"
    return f"""## Fragment
{text}

## Annotation A
strategy1: {a_s1}
strategy2: {a_s2}
explanation: {a_expl}

## Annotation B
strategy1: {b_s1}
strategy2: {b_s2}
explanation: {b_expl}
"""

In [9]:
TIE_BREAK_TOOL_ANTHROPIC = {
    "name": "choose_annotation",
    "description": "Choose which of the two annotations (A or B) is more accurate.",
    "input_schema": {
        "type": "object",
        "properties": {
            "choice": {"type": "string", "enum": ["A", "B"]},
            "explanation": {"type": "string"}
        },
        "required": ["choice", "explanation"],
        "additionalProperties": False
    }
}

def tie_break_anthropic(user_msg, model=ANTHROPIC_MODEL):
    response = anthropic_client.messages.create(
        model=model,
        max_tokens=300,
        temperature=0,
        system=[
            {
                "type": "text",
                "text": TIE_BREAK_PROMPT,
                "cache_control": {"type": "ephemeral"}
            }
        ],
        messages=[{"role": "user", "content": user_msg}],
        tools=[TIE_BREAK_TOOL_ANTHROPIC],
        tool_choice={"type": "tool", "name": "choose_annotation"}
    )
    tool_use = next(b for b in response.content if b.type == "tool_use")
    return dict(tool_use.input)

In [10]:
cols_for_third_model = [
    "title", "link", "election_type", "date", "speaker", "place",
    "text", "order",
    "strategy1", "strategy2",
    "cohere_strategy1", "cohere_strategy2",
    "cohere_confidence", "cohere_explanation",
    "mistral_strategy1", "mistral_strategy2",
    "mistral_confidence", "mistral_explanation",
]

disputed[cols_for_third_model].to_csv("disputed_for_third_model.csv", sep=";", index=False)
print(f"Сохранено: disputed_for_third_model.csv ({len(disputed)} строк)")

Сохранено: disputed_for_third_model.csv (7367 строк)


In [15]:
ANTHROPIC_CHECKPOINT = "disputed_anthropic_annotated.csv"
SAVE_EVERY = 100

if os.path.exists(ANTHROPIC_CHECKPOINT):
    print(f"Найден чекпоинт {ANTHROPIC_CHECKPOINT}, продолжаем.")
    disputed_annotated = pd.read_csv(ANTHROPIC_CHECKPOINT, sep=";")
    done = disputed_annotated["anthropic_choice"].notna().sum()
    print(f"Уже размечено: {done}")
else:
    disputed_annotated = disputed.copy()
    disputed_annotated["anthropic_choice"] = None
    disputed_annotated["anthropic_explanation"] = None
    disputed_annotated["anthropic_error"] = None

todo = disputed_annotated[disputed_annotated["anthropic_choice"].isna()].index.tolist()
print(f"Осталось обработать: {len(todo)}")

Найден чекпоинт disputed_anthropic_annotated.csv, продолжаем.
Уже размечено: 7343
Осталось обработать: 24


In [16]:
completed_since_save = 0

MAX_WORKERS = 8
completed_since_save = 0

def process_one(idx):
    row = disputed_annotated.loc[idx]
    user_msg = build_tie_break_user_message(
        text=row["text"],
        a_s1=row["cohere_strategy1"], a_s2=row["cohere_strategy2"], a_expl=row["cohere_explanation"],
        b_s1=row["mistral_strategy1"], b_s2=row["mistral_strategy2"], b_expl=row["mistral_explanation"],
    )
    try:
        result = tie_break_anthropic(user_msg)
        return idx, result, None
    except Exception as e:
        return idx, None, str(e)

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(process_one, idx): idx for idx in todo}
    pbar = tqdm(total=len(futures), desc=f"Anthropic ({ANTHROPIC_MODEL})")
    try:
        for future in as_completed(futures):
            idx, result, err = future.result()
            if result is not None:
                disputed_annotated.at[idx, "anthropic_choice"] = result.get("choice")
                disputed_annotated.at[idx, "anthropic_explanation"] = result.get("explanation")
                disputed_annotated.at[idx, "anthropic_error"] = None
            else:
                disputed_annotated.at[idx, "anthropic_error"] = err

            completed_since_save += 1
            pbar.update(1)

            if completed_since_save >= SAVE_EVERY:
                disputed_annotated.to_csv(ANTHROPIC_CHECKPOINT, sep=";", index=False)
                completed_since_save = 0
    except KeyboardInterrupt:
        print("\nInterrupted by user. Saving progress...")
    finally:
        pbar.close()
        disputed_annotated.to_csv(ANTHROPIC_CHECKPOINT, sep=";", index=False)

print(f"\nГотово. Ошибок: {disputed_annotated['anthropic_error'].notna().sum()}")
print(f"Без выбора: {disputed_annotated['anthropic_choice'].isna().sum()}")
print(f"Сохранено: {ANTHROPIC_CHECKPOINT}")

Anthropic (claude-sonnet-4-5): 100%|██████████| 24/24 [00:13<00:00,  1.75it/s]


Готово. Ошибок: 0
Без выбора: 0
Сохранено: disputed_anthropic_annotated.csv


In [17]:
merged["ensemble_strategy1"] = merged["cohere_strategy1"]
merged["ensemble_strategy2"] = merged["cohere_strategy2"]
merged["ensemble_source"]    = "consensus"

anth_key_cols = ["text", "speaker", "order"]
anth_lookup = disputed_annotated.set_index(anth_key_cols)[
    ["anthropic_choice", "anthropic_explanation"]
]

n_a = n_b = n_missing = 0
for idx in merged[~agree_both].index:
    key = tuple(merged.loc[idx, anth_key_cols])
    if key not in anth_lookup.index:
        n_missing += 1
        continue
    choice = anth_lookup.loc[key, "anthropic_choice"]
    if choice == "A":
        merged.at[idx, "ensemble_source"] = "anthropic->cohere"
        n_a += 1
    elif choice == "B":
        merged.at[idx, "ensemble_strategy1"] = merged.at[idx, "mistral_strategy1"]
        merged.at[idx, "ensemble_strategy2"] = merged.at[idx, "mistral_strategy2"]
        merged.at[idx, "ensemble_source"]    = "anthropic->mistral"
        n_b += 1
    else:
        merged.at[idx, "ensemble_source"] = "anthropic_failed"
        n_missing += 1

print(f"Anthropic выбрал A (Cohere):  {n_a}")
print(f"Anthropic выбрал B (Mistral): {n_b}")
print(f"Без ответа / ошибки:          {n_missing}")
print(f"\nensemble_source распределение:")
print(merged["ensemble_source"].value_counts())

Anthropic выбрал A (Cohere):  3623
Anthropic выбрал B (Mistral): 3744
Без ответа / ошибки:          0

ensemble_source распределение:
ensemble_source
consensus             34917
anthropic->mistral     3744
anthropic->cohere      3623
Name: count, dtype: int64
